# Quick SLM — 10 · Second judge (self-preference bias)

The evaluation in notebook 07 scores the model with Gemma 4, which also generated
the fine-tuning corpus. A model tends to score text distributed like its own
output favourably, so those scores are biased upward by an unknown amount. This
notebook measures that bias instead of conceding it: it re-scores the **same
cached responses** with a judge from a different family and reports the gap.

Nothing is generated here. It reads the `sft_probe_outputs_*.json` written by 07,
scores each response with a non-Gemma judge under the byte-identical rubric, and
compares against the Gemma scores. The judge is set in Section 5 (`JUDGE_MODEL`);
use any capable instruct model of a different family from the teacher, loaded in
4-bit nf4.

The number that matters is the per-category gap between the two judges, and
whether the grounding conclusion of Section 10 survives an independent judge.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies

In [ ]:
!pip -q install --upgrade transformers accelerate safetensors tokenizers tqdm pandas bitsandbytes

## 3. Locate the repository and check framework support

In [ ]:
import sys, json, re
from pathlib import Path

REPO_DIR = Path('/content/drive/MyDrive/quick-slm/code')
framework_dir = REPO_DIR / 'framework'
assert framework_dir.is_dir(), f'{framework_dir} not found'
if str(framework_dir) not in sys.path:
    sys.path.insert(0, str(framework_dir))

from v1.quick_slm_trainer.support import require_framework
require_framework('v1', REPO_DIR)
from v1.quick_slm_trainer.paths import Layout

DRIVE_ROOT = Path('/content/drive/MyDrive/quick-slm')
LOGS_DIR = DRIVE_ROOT / 'logs'
layout = Layout(drive_root=DRIVE_ROOT)
print('framework OK')

## 4. Load the cached generations

These are the exact responses notebook 07 scored with Gemma. Re-scoring them,
rather than regenerating, is what makes the two judges comparable: same model
outputs, different grader.

In [ ]:
LABELS = ['base', 'sft-final']

def probe_path(label):  return LOGS_DIR / f'sft_probe_outputs_{label}.json'
def gemma_path(label):  return LOGS_DIR / f'sft_scored_outputs_{label}.json'
def out_path(label):    return LOGS_DIR / f'sft_scored_outputs_{label}_second.json'

cached = {}
for lab in LABELS:
    p = probe_path(lab)
    assert p.exists(), f'{p} not found -- run notebook 07 section 6 first'
    cached[lab] = json.loads(p.read_text())
    print(f'{lab:<10} {len(cached[lab]["outputs"]):>5,} cached generations')

## 5. Load the second judge

A different family from Gemma, which is the whole point. Same 4-bit setup as 06
and 07 so it fits a single GPU.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# A non-Gemma judge, loaded in 4-bit nf4. This path takes a full-precision
# checkpoint and quantizes it at load; do NOT point it at a pre-quantized NVFP4
# checkpoint (those need vLLM/modelopt, not bitsandbytes). Verify the exact id on
# the model's HuggingFace page before running -- it must resolve as org/name.
JUDGE_MODEL = 'Qwen/Qwen3.6-27B'

# Reasoning models emit a chain of thought before the answer. With a short
# generation budget the SCORE: line is never reached and every reply parses as
# -1; that is exactly what a first run of this notebook produced, 1,386 unparsed
# out of 1,386. Two defences, both cheap:
#   DISABLE_THINKING asks the chat template to skip the thinking block, which
#   Qwen3-family templates support via enable_thinking=False and others ignore.
#   MAX_NEW is then large enough to hold a thinking block anyway if the template
#   does not honour the flag.
DISABLE_THINKING = True
MAX_NEW = 512

_q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                        bnb_4bit_compute_dtype=torch.bfloat16,
                        bnb_4bit_use_double_quant=True)
print(f'loading judge: {JUDGE_MODEL} (4-bit nf4)')
jtok = AutoTokenizer.from_pretrained(JUDGE_MODEL)
jmodel = AutoModelForCausalLM.from_pretrained(JUDGE_MODEL, quantization_config=_q,
                                              device_map='auto').eval()
print(f'  loaded -- {sum(p.numel() for p in jmodel.parameters())/1e9:.1f}B params')

# Byte-identical to notebook 07's JUDGE_RUBRIC, so a score here means the same
# thing on the same 0-5 scale.
JUDGE_RUBRIC = """You are evaluating one output from a small (103M parameter) tool-calling model that has just been fine-tuned.

The model is given a user request and a set of tools. A correct response reasons briefly, then emits the tool call shown under EXPECTED -- the same tool, with the same argument values. Judge the substance of the call, not its punctuation or whitespace.

Score the response from 0 to 5:
- 0: empty, repetition, or no tool call at all
- 1: a tool call, but unrelated to the request
- 2: plausible tool, wrong task; or the right tool with an argument the user never supplied
- 3: right tool, wrong argument value
- 4: right tool and right arguments, with a flaw in the reasoning or the formatting
- 5: the expected call, with reasoning that supports it

If a STATE block appears in the prompt it is authoritative and outranks any MEMORY block. A response that follows memory against a conflicting state is wrong, however fluent it reads.

Be strict. This is a 103M model and most responses should score 0-2.

PROMPT:
{prompt}

EXPECTED (the call a correct response makes):
{expected}

MODEL RESPONSE:
{response}

Respond with EXACTLY this format and nothing else:
SCORE: <integer 0-5>
REASON: <one short sentence>"""

_SCORE_RE  = re.compile(r'SCORE:\s*([0-5])\b', re.IGNORECASE)
_REASON_RE = re.compile(r'REASON:\s*(.+)', re.IGNORECASE)

def _apply_template(msg):
    kw = dict(tokenize=False, add_generation_prompt=True)
    if DISABLE_THINKING:
        try:
            return jtok.apply_chat_template([{'role': 'user', 'content': msg}],
                                            enable_thinking=False, **kw)
        except TypeError:
            pass          # template does not take the flag; fall through
    return jtok.apply_chat_template([{'role': 'user', 'content': msg}], **kw)

def _parse(reply):
    # A reasoning model may mention the rubric's numbers while thinking, so take
    # the LAST SCORE: in the reply, which is the one it committed to.
    body = reply.rsplit('</think>', 1)[-1]        # drop any thinking block
    hits = _SCORE_RE.findall(body) or _SCORE_RE.findall(reply)
    if not hits:
        return -1, reply.strip()[-400:]           # keep the TAIL: that is where
                                                  # the answer would have been
    r = _REASON_RE.search(body) or _REASON_RE.search(reply)
    return int(hits[-1]), (r.group(1).strip().split('\n')[0][:200] if r else '')

@torch.no_grad()
def judge_score(prompt, expected, response):
    msg = JUDGE_RUBRIC.format(prompt=prompt[:800], expected=expected, response=response[:800])
    enc = jtok(_apply_template(msg), return_tensors='pt').to(jmodel.device)
    out = jmodel.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                          pad_token_id=jtok.eos_token_id)
    reply = jtok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True)
    return _parse(reply)

# Smoke-test on three real cases before paying for 2,772. If this prints -1, the
# format is wrong and the full run would waste an hour producing nothing.
_probe = cached[LABELS[-1]]['outputs'][:3]
print('\nsmoke test:')
for _o in _probe:
    _s, _r = judge_score(_o['prompt'], _o['expected'], _o['response'])
    print(f'  score {_s:>3}   {_r[:90]}')
assert any(judge_score(o['prompt'], o['expected'], o['response'])[0] >= 0 for o in _probe), (
    'the judge produced no parseable score on any probe. Raise MAX_NEW, check '
    'DISABLE_THINKING, or inspect a raw reply before running the full pass.')
print('  -> parses; safe to run section 6')

## 6. Re-score every cached response

In [ ]:
from tqdm.auto import tqdm

scored = {}
for lab in LABELS:
    recs = cached[lab]['outputs']
    print(f'scoring {lab} ({len(recs):,})')
    out = {'label': lab, 'judge': JUDGE_MODEL, 'outputs': []}
    for i, o in enumerate(tqdm(recs, desc=f'  {lab}', leave=False)):
        s, why = judge_score(o['prompt'], o['expected'], o['response'])
        out['outputs'].append({**o, 'score': s, 'reason': why})
        # Bail out early rather than spend an hour producing -1s. The first run
        # of this notebook returned 1,386 unparsed replies because the judge's
        # thinking block ate the generation budget; this stops that at 50.
        if i == 49 and all(r['score'] < 0 for r in out['outputs']):
            raise RuntimeError(
                'first 50 replies all unparsed -- stopping. Inspect '
                "out['outputs'][0]['reason'] (it holds the tail of the raw reply), "
                'then raise MAX_NEW or check DISABLE_THINKING in section 5.')
    out_path(lab).write_text(json.dumps(out, indent=2, default=str))
    scored[lab] = out
    unparsed = sum(1 for o in out['outputs'] if o['score'] < 0)
    print(f'  wrote {out_path(lab).name}   unparsed {unparsed:,} / {len(recs):,}'
          f'  ({unparsed/len(recs):.1%})')
    if unparsed > len(recs) * 0.05:
        print('  WARNING: more than 5% unparsed; the comparison rests on the rest.')

## 7. The two judges, side by side

The gap between the columns is the self-preference bias, measured. A small gap
means Gemma was not much flattering its own student; a large positive Gemma minus
second-judge gap means the Section 10 scores were inflated. The grounding rows
are the ones the paper turns on.

In [ ]:
import pandas as pd
from collections import defaultdict

def cat_means(outs):
    ok = [o for o in outs if o['score'] >= 0]
    m = {}
    for c in sorted({o['category'] for o in ok}):
        xs = [o['score'] for o in ok if o['category'] == c]
        m[c] = sum(xs)/len(xs)
    m['ALL'] = sum(o['score'] for o in ok)/len(ok)
    return m

def grounded(outs):
    pairs = defaultdict(list)
    for o in outs:
        if o['category']=='state_memory_conflict' and o.get('group') and o['score']>=0:
            pairs[o['group']].append(o['score'])
    comp = {g:v for g,v in pairs.items() if len(v)==2}
    return sum(1 for v in comp.values() if min(v)>=4), len(comp)

for lab in LABELS:
    gemma = json.loads(gemma_path(lab).read_text())['outputs'] if gemma_path(lab).exists() else None
    second = scored[lab]['outputs']
    print(f'\n=== {lab} ===')
    if gemma:
        gm, sm = cat_means(gemma), cat_means(second)
        rows = [{'category': c, 'Gemma': round(gm.get(c, float("nan")),2),
                 JUDGE_MODEL.split("/")[-1]: round(sm.get(c, float("nan")),2),
                 'gap': round(gm.get(c,0)-sm.get(c,0),2)} for c in sm]
        print(pd.DataFrame(rows).to_string(index=False))
        gb, gn = grounded(gemma); sb, sn = grounded(second)
        print(f'grounded pairs: Gemma {gb}/{gn}   {JUDGE_MODEL.split("/")[-1]} {sb}/{sn}')
        # agreement: correlation of per-item scores where both parsed
        both = [(g["score"], s["score"]) for g, s in zip(gemma, second)
                if g["score"]>=0 and s["score"]>=0]
        if both:
            import statistics
            gs = [a for a,_ in both]; ss = [b for _,b in both]
            mg, ms = statistics.fmean(gs), statistics.fmean(ss)
            cov = sum((a-mg)*(b-ms) for a,b in both)
            den = (sum((a-mg)**2 for a in gs)*sum((b-ms)**2 for b in ss))**0.5
            print(f'per-item score correlation: {cov/den:.3f}  (n={len(both):,})')
            print(f'mean |Gemma - second| per item: {sum(abs(a-b) for a,b in both)/len(both):.2f}')
    else:
        print('no Gemma scores on disk to compare; run notebook 07 first')
        print(cat_means(second))

## 8. Save the comparison

In [ ]:
summary = {'judge_second': JUDGE_MODEL, 'judge_first': 'google/gemma-4-31B-it-qat-q4_0-unquantized',
           'labels': LABELS,
           'note': 'second judge re-scores 07 cached generations; gap measures self-preference bias'}
(LOGS_DIR / 'second_judge_summary.json').write_text(json.dumps(summary, indent=2))
print('per-label scores in', LOGS_DIR)
for lab in LABELS: print('  ', out_path(lab).name)
print('\nSend both sft_scored_outputs_*_second.json for the paper.')